# Project: automating an instrument

Read a (simulated) instrument, store its measurements in an SQLite database, query them and plot them.

The full explanations are in [instrument_logging.md](instrument_logging.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Step 1: a (simulated) instrument

The reading of the instrument is isolated in **one function**,
`read_temperature`: to use a real instrument, only this function has to be
rewritten. The module also contains the functions to work with the database
(`sqlite3` is part of Python's standard library).

In [ ]:
%%writefile instrument.py
"""Simulated instrument and storage of its measurements in an SQLite database."""

import sqlite3

import numpy as np


def read_temperature(rng, elapsed):
    """Simulate the reading of a temperature probe in a heated reactor.

    Parameters
    ----------
    rng : numpy.random.Generator
        Random generator used for the measurement noise.
    elapsed : float
        Time since the start of the heating, in seconds.

    Returns
    -------
    float
        Temperature in °C: rises from 20 °C towards 80 °C, plus noise.
    """
    return 20 + 60 * (1 - np.exp(-elapsed / 5)) + rng.normal(0, 0.3)


def create_database(filename):
    """Open (or create) the database and make sure the table exists."""
    connection = sqlite3.connect(filename)
    connection.execute(
        "CREATE TABLE IF NOT EXISTS measurements "
        "(timestamp TEXT, elapsed REAL, temperature REAL)")
    connection.commit()
    return connection


def save_measurement(connection, timestamp, elapsed, temperature):
    """Store one measurement in the database."""
    connection.execute("INSERT INTO measurements VALUES (?, ?, ?)",
                       (timestamp, elapsed, temperature))
    connection.commit()


def load_measurements(connection):
    """Return all measurements as a 2D array: elapsed time, temperature."""
    rows = connection.execute(
        "SELECT elapsed, temperature FROM measurements ORDER BY elapsed").fetchall()
    return np.array(rows)

In [ ]:
import numpy as np
from instrument import read_temperature

rng = np.random.default_rng(0)
print(read_temperature(rng, 0))     # about 20 °C
print(read_temperature(rng, 30))    # close to 80 °C

## Step 2: tests

`tmp_path` is a pytest *fixture*: a temporary folder created for the test,
so that the test does not touch your real database.

In [ ]:
%%writefile test_instrument.py
"""Tests for instrument.py."""

import numpy as np

from instrument import (create_database, load_measurements, read_temperature,
                        save_measurement)


def test_temperature_starts_near_20():
    rng = np.random.default_rng(0)
    assert abs(read_temperature(rng, 0) - 20) < 2


def test_temperature_tends_to_80():
    rng = np.random.default_rng(0)
    assert abs(read_temperature(rng, 1000) - 80) < 2


def test_database_round_trip(tmp_path):
    connection = create_database(tmp_path / "test.db")
    save_measurement(connection, "2026-01-01T10:00:00", 0.0, 20.5)
    save_measurement(connection, "2026-01-01T10:00:01", 1.0, 25.0)
    data = load_measurements(connection)
    assert data.shape == (2, 2)
    assert data[1, 1] == 25.0

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: the acquisition script

The script reads the instrument at regular intervals, stores each
measurement and warns above an alarm level. Here we run a short
acquisition (20 s); use the *stop* button (■) of the toolbar to interrupt
it.

In [ ]:
%%writefile acquire.py
"""Acquire measurements from the (simulated) instrument and store them."""

import argparse
import time
from datetime import datetime

import numpy as np

from instrument import create_database, read_temperature, save_measurement


def main():
    parser = argparse.ArgumentParser(description="Temperature data logger.")
    parser.add_argument("--duration", type=float, default=20,
                        help="acquisition duration (s)")
    parser.add_argument("--interval", type=float, default=0.5,
                        help="time between two measurements (s)")
    parser.add_argument("--alarm", type=float, default=75,
                        help="print a warning above this temperature (°C)")
    parser.add_argument("--db", default="measurements.db",
                        help="database file")
    args = parser.parse_args()

    rng = np.random.default_rng()
    connection = create_database(args.db)
    start = time.time()
    elapsed = 0.0
    while elapsed < args.duration:
        temperature = read_temperature(rng, elapsed)
        timestamp = datetime.now().isoformat(timespec="seconds")
        save_measurement(connection, timestamp, elapsed, temperature)
        message = f"{timestamp}  t = {elapsed:5.1f} s  T = {temperature:5.1f} °C"
        if temperature > args.alarm:
            message += "  ALARM!"
        print(message)
        time.sleep(args.interval)
        elapsed = time.time() - start
    connection.close()
    print("Data saved in", args.db)


if __name__ == "__main__":
    main()

In [ ]:
%run acquire.py --duration 20 --interval 0.5

Run the cell again: the new measurements are **added** to the same database.

## Step 4: querying the database in SQL

In [ ]:
import sqlite3

connection = sqlite3.connect("measurements.db")
print(connection.execute("SELECT COUNT(*), AVG(temperature), MAX(temperature) "
                         "FROM measurements").fetchone())
print(connection.execute("SELECT timestamp, temperature FROM measurements "
                         "WHERE temperature > 75 LIMIT 3").fetchall())
connection.close()

## Step 5: plotting

In [ ]:
%%writefile plot_measurements.py
"""Plot the measurements stored in the database."""

import argparse

import matplotlib.pyplot as plt

from instrument import create_database, load_measurements


def main():
    parser = argparse.ArgumentParser(description="Plot logged temperatures.")
    parser.add_argument("--db", default="measurements.db", help="database file")
    parser.add_argument("--alarm", type=float, default=75, help="alarm level (°C)")
    args = parser.parse_args()

    connection = create_database(args.db)
    data = load_measurements(connection)
    connection.close()
    elapsed, temperature = data[:, 0], data[:, 1]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(elapsed, temperature, ".-")
    ax.axhline(args.alarm, color="red", linestyle="--", label="alarm")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Temperature (°C)")
    ax.legend()
    fig.savefig("measurements.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run plot_measurements.py

## Using a real instrument

Serial instruments: `conda install pyserial`; SCPI instruments (USB, GPIB,
Ethernet): `conda install pyvisa pyvisa-py`. See the lesson for examples. The
commands to send are given in the manual of the instrument.

## Going further

**1.** Modify the alarm so that `ALARM!` is printed only when the temperature **crosses** the threshold, and `back to normal` when it goes back below.

<details><summary>Answer</summary>

Keep the state in a variable created before the loop:

```python
    alarm_on = False
    while elapsed < args.duration:
        ...
        if temperature > args.alarm and not alarm_on:
            print("ALARM!")
            alarm_on = True
        elif temperature <= args.alarm and alarm_on:
            print("back to normal")
            alarm_on = False
```

</details>

In [ ]:
# Your code here

**2.** Export the whole database into a CSV file.

<details><summary>Answer</summary>

```python
import csv
import sqlite3

connection = sqlite3.connect("measurements.db")
rows = connection.execute("SELECT * FROM measurements").fetchall()
connection.close()
with open("measurements.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "elapsed", "temperature"])
    writer.writerows(rows)
```

</details>

In [ ]:
# Your code here

**3.** Make the acquisition robust: if `read_temperature` raises an error, print a message, skip this measurement and continue (`try` / `except`).

<details><summary>Answer</summary>

```python
        try:
            temperature = read_temperature(rng, elapsed)
        except Exception as error:
            print("Reading failed:", error)
            time.sleep(args.interval)
            elapsed = time.time() - start
            continue
```

</details>

In [ ]:
# Your code here